In [1]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importamos nuestro módulo de ML
ml = importlib.import_module("flows.05_ml_models")
sns.set_theme(style="whitegrid", context="notebook")

# Cargamos el dataset formulado
X, y = ml.load_dataset()
print("Dimensiones de X:", X.shape)
print("Tasa de supervivencia (y.mean()):", round(y.mean(), 4))

Dimensiones de X: (891, 12)
Tasa de supervivencia (y.mean()): 0.3838


# Formulación del Problema de Machine Learning (Paso 39)

| Decisión | Tu respuesta |
| :--- | :--- |
| **Tipo de problema** | Clasificación binaria supervisada |
| **Variable objetivo** | `Survived` (1 = sobrevivió) |
| **Unidad de análisis y momento de predicción** | Un pasajero, *al embarcar* |
| **Métrica principal** | ROC-AUC, porque evalúa la capacidad de discriminación sin depender de un umbral de decisión rígido en un entorno desbalanceado (~38% positivos). |
| **Métricas secundarias** | precisión, recall, exactitud, F1-score |
| **Costo de errores** | Para una aseguradora, predecir falsamente que alguien sobrevive genera un riesgo financiero crítico de estimación; predecir que muere representa una oportunidad comercial perdida. |
| **Criterio de éxito** | Superar el baseline por un margen que puedas defender |


## Paso 40. Auditoría de variables y fuga de datos

| Columna | ¿Existe antes del evento? | ¿Riesgo de fuga? | Decisión |
| :--- | :--- | :--- | :--- |
| `Survived`, `Survivor` | No (es el objetivo) | **Fuga directa** | **Excluir** |
| `PassengerId`, `Ticket`, `Name` | Sí | Identificadores: memorizan, no generalizan | **Excluir** (el título sí, vía `TTitulo`) |
| `NSex`, `NEmbarked`, `NCabin`, `NTitulo` | — | Duplican a `Sex`, etc. con un orden arbitrario | **Excluir** (usar *one-hot*) |
| `Age` imputada | — | Estadística calculada con todos los datos | **Usar `AgeOriginal`** |
| `Cabin`, `TCabin`, `HasCabin` | **Dudoso** | ¿Se registró la cabina *después* del naufragio (más fácil para los sobrevivientes)? | **Evaluar con y sin ella** (Paso 61) |
| `Person` | Derivada de Age / Sex | Redundante con ambas | **Excluir del modelo base** |

**Reflexión sobre el debate de `HasCabin`:**
Si `HasCabin` mejora la métrica de predicción, no significa necesariamente que tener cabina otorgue una ventaja física de supervivencia. Existe un **artefacto de registro**: la información de la cabina se conservó y documentó con mayor precisión para los pasajeros que sobrevivieron y pudieron dar testimonio, inflando artificialmente el valor predictivo de esta variable. Un modelo robusto debe evaluar este tipo de variables con cautela para no aprender sesgos burocráticos en lugar de riesgo real.

In [4]:
import importlib

# Como el archivo empieza con número, se importa estrictamente con importlib:
ml = importlib.import_module("flows.05_ml_models")
importlib.reload(ml)

# Cargamos el dataset y realizamos la división estratificada
X, y = ml.load_dataset()
X_train, X_test, y_train, y_test = ml.split(X, y)

print("Dimensiones de Train:", X_train.shape)
print("Dimensiones de Test:", X_test.shape)
print("-" * 40)
print("Tasa de supervivencia en Train:", round(y_train.mean(), 4))
print("Tasa de supervivencia en Test: ", round(y_test.mean(), 4))

Dimensiones de Train: (712, 12)
Dimensiones de Test: (179, 12)
----------------------------------------
Tasa de supervivencia en Train: 0.3834
Tasa de supervivencia en Test:  0.3855


In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- 1. Baseline "Ingenuo" (DummyClassifier) ---
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_train)

print("=== Baseline 1: DummyClassifier (Clase Mayoritaria) ===")
print(f"Exactitud (Accuracy): {accuracy_score(y_train, y_pred_dummy):.4f}")
print("-" * 50)

# --- 2. Baseline "Regla de Género" (Mujeres sobreviven, hombres no) ---
# Predice 1 si Sex == 'female', 0 en caso contrario
y_pred_gender = (X_train['Sex'] == 'female').astype(int)

acc_g = accuracy_score(y_train, y_pred_gender)
prec_g = precision_score(y_train, y_pred_gender)
rec_g = recall_score(y_train, y_pred_gender)
f1_g = f1_score(y_train, y_pred_gender)

print("=== Baseline 2: Regla de Género (Sex == 'female') ===")
print(f"Exactitud (Accuracy): {acc_g:.4f}  (Debería rondar ~0.79)")
print(f"Precisión:            {prec_g:.4f}")
print(f"Recall (Sensibilidad): {rec_g:.4f}")
print(f"F1-score:             {f1_g:.4f}")

=== Baseline 1: DummyClassifier (Clase Mayoritaria) ===
Exactitud (Accuracy): 0.6166
--------------------------------------------------
=== Baseline 2: Regla de Género (Sex == 'female') ===
Exactitud (Accuracy): 0.7893  (Debería rondar ~0.79)
Precisión:            0.7431
Recall (Sensibilidad): 0.6886
F1-score:             0.7148


In [6]:
# --- Paso 43: Experimento de Fuga de Datos (Data Leakage) ---

# 1. Edad media de las mujeres usando TODO el dataset (X)
# (Esto simula el error de imputar o calcular estadísticas antes del split)
age_mean_all = X[X['Sex'] == 'female']['Age'].mean()

# 2. Edad media de las mujeres usando SOLO el conjunto de entrenamiento (X_train)
# (Esto es lo correcto: el test set permanece completamente ciego)
age_mean_train = X_train[X_train['Sex'] == 'female']['Age'].mean()

print(f"Edad media mujeres (TODO el dataset X):  {age_mean_all:.4f}")
print(f"Edad media mujeres (SOLO X_train):      {age_mean_train:.4f}")
print(f"Diferencia absoluta:                  {abs(age_mean_all - age_mean_train):.4f}")

Edad media mujeres (TODO el dataset X):  27.9157
Edad media mujeres (SOLO X_train):      28.1834
Diferencia absoluta:                  0.2677


**Reflexión del experimento de Fuga de Datos (Paso 43):**
1. **¿Qué información viajó?** Información estadística sobre la distribución de la edad del conjunto de prueba (`X_test`) se filtró prematuramente hacia los cálculos de entrenamiento al usar todo el dataset.
2. **¿Por qué la diferencia es pequeña y por qué es un error?** La diferencia es pequeña aquí por el gran tamaño de la muestra y la estabilidad del Titanic, pero metodológicamente es un error grave: en otros datasets con muestras pequeñas o alta proporción de nulos, esta filtración infla artificialmente el rendimiento del modelo y destruye su capacidad de generalización real.